# NGLab Tutorial #6: Hyperparameter Optimization

Master DEHB (Differential Evolution Hyperband) for efficient hyperparameter search.

## Learning Objectives

1. Understand successive halving and multi-fidelity optimization
2. Define search spaces for ML models
3. Run DEHB optimization
4. Visualize convergence and best configurations

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution

np.random.seed(42)
print("Libraries loaded!")

## 1. The DEHB Algorithm

DEHB combines:
1. **Hyperband**: Resource allocation via successive halving
2. **Differential Evolution**: Intelligent mutation for new configs

### Successive Halving

```
Round 1: Train 64 configs for 10 epochs → Keep top 16
Round 2: Train 16 configs for 30 epochs → Keep top 4
Round 3: Train 4 configs for 100 epochs → Keep top 1
```

### Differential Evolution Mutation

$$
v_i = x_{r1} + F \cdot (x_{r2} - x_{r3})
$$

In [ ]:
# Define search space
search_space = {
    'learning_rate': (1e-5, 1e-2, 'log'),  # Log-uniform
    'batch_size': (16, 256, 'int'),         # Integer
    'hidden_dim': (64, 512, 'int'),
    'dropout': (0.0, 0.5, 'uniform'),       # Uniform
    'weight_decay': (1e-6, 1e-3, 'log'),
}

def sample_config(space):
    """Sample a random configuration."""
    config = {}
    for param, (low, high, scale) in space.items():
        if scale == 'log':
            config[param] = 10 ** np.random.uniform(np.log10(low), np.log10(high))
        elif scale == 'int':
            config[param] = int(np.random.randint(low, high + 1))
        else:  # uniform
            config[param] = np.random.uniform(low, high)
    return config

# Sample some configs
print("Example configurations:")
for i in range(3):
    print(f"  Config {i+1}: {sample_config(search_space)}")

## 2. Mock Objective Function

In [ ]:
def objective_function(config, fidelity=100):
    """Simulated training objective.
    
    Args:
        config: Hyperparameter configuration
        fidelity: Training budget (num epochs)
    
    Returns:
        Loss value (lower is better)
    """
    # Optimal config (unknown to optimizer)
    optimal = {
        'learning_rate': 3e-4,
        'batch_size': 128,
        'hidden_dim': 256,
        'dropout': 0.2,
        'weight_decay': 1e-5
    }
    
    # Compute normalized distance from optimal
    distance = 0
    distance += (np.log10(config['learning_rate']) - np.log10(optimal['learning_rate']))**2
    distance += ((config['batch_size'] - optimal['batch_size']) / 100)**2
    distance += ((config['hidden_dim'] - optimal['hidden_dim']) / 100)**2
    distance += (config['dropout'] - optimal['dropout'])**2 * 10
    distance += (np.log10(config['weight_decay']) - np.log10(optimal['weight_decay']))**2
    
    # Base loss
    base_loss = 0.5 + distance / 10
    
    # Fidelity effect (more epochs = better estimate)
    noise_scale = 0.1 / np.sqrt(fidelity / 10)
    noise = np.random.randn() * noise_scale
    
    return base_loss + noise

# Test
test_config = sample_config(search_space)
print(f"\nTest config: {test_config}")
print(f"Loss @10 epochs: {objective_function(test_config, 10):.4f}")
print(f"Loss @100 epochs: {objective_function(test_config, 100):.4f}")

## 3. Successive Halving Schedule

In [ ]:
def successive_halving(n_configs=64, max_fidelity=100, eta=4):
    """Generate SH schedule."""
    schedule = []
    n = n_configs
    fidelity = max_fidelity // (eta ** int(np.log(n_configs) / np.log(eta)))
    
    while n >= 1:
        schedule.append({'n_configs': int(n), 'fidelity': int(fidelity)})
        n /= eta
        fidelity *= eta
    
    return schedule

schedule = successive_halving()
print("\nSuccessive Halving Schedule:")
for i, rung in enumerate(schedule):
    print(f"  Rung {i+1}: Train {rung['n_configs']:2d} configs for {rung['fidelity']:3d} epochs")

## 4. Run Optimization

In [ ]:
# Run simplified DEHB
history = []
best_loss = float('inf')
best_config = None

for bracket in range(3):  # 3 brackets
    print(f"\n=== Bracket {bracket + 1} ===")
    configs = [sample_config(search_space) for _ in range(schedule[0]['n_configs'])]
    
    for rung in schedule:
        # Evaluate all configs at this fidelity
        results = []
        for cfg in configs:
            loss = objective_function(cfg, rung['fidelity'])
            results.append((loss, cfg))
            history.append({
                'bracket': bracket,
                'fidelity': rung['fidelity'],
                'loss': loss,
                'config': cfg
            })
            
            if loss < best_loss:
                best_loss = loss
                best_config = cfg.copy()
        
        # Sort and keep top performers
        results.sort(key=lambda x: x[0])
        n_keep = max(1, len(results) // 4)
        configs = [cfg for _, cfg in results[:n_keep]]
        
        print(f"  Fidelity {rung['fidelity']:3d}: Best loss = {results[0][0]:.4f}")

print(f"\n=== Optimization Complete ===")
print(f"Best loss: {best_loss:.4f}")
print(f"Best config: {best_config}")

In [ ]:
# Visualize convergence
history_df = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss over evaluations
cummin_loss = np.minimum.accumulate(history_df['loss'].values)
ax1.plot(cummin_loss, linewidth=2, color='blue')
ax1.set_title('Best Loss Over Evaluations', fontsize=12, fontweight='bold')
ax1.set_xlabel('Evaluation')
ax1.set_ylabel('Best Loss')
ax1.grid(True, alpha=0.3)

# Loss by fidelity
for fid in sorted(history_df['fidelity'].unique()):
    subset = history_df[history_df['fidelity'] == fid]
    ax2.scatter(range(len(subset)), subset['loss'], label=f'{fid} epochs', alpha=0.6)

ax2.set_title('Loss by Fidelity Level', fontsize=12, fontweight='bold')
ax2.set_xlabel('Configuration Index')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In this notebook, you learned:

✅ DEHB algorithm fundamentals  
✅ Successive halving for resource allocation  
✅ Search space definition  
✅ Convergence visualization  

## Next Steps

Continue to **Notebook #8**: Multi-Agent Simulation!

---